In [2]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Config ===
INPUT_PATH = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_removal_keyword_check_output.csv"
OUTPUT_PATH = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step6_ci_detection_output.csv"
SAVE_INTERVAL = 20
SLEEP_TIME = 0.2

# === Load Tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0
def get_auth_header():
    global token_index
    header = {"Authorization": f"token {tokens[token_index]}"}
    token_index = (token_index + 1) % len(tokens)
    return header

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}
compiled_patterns = [(re.compile(p), name) for p, name in ci_patterns.items()]
ci_services = sorted(set(ci_patterns.values()))

# === Load Data ===
if os.path.exists(OUTPUT_PATH):
    df = pd.read_csv(OUTPUT_PATH)
else:
    df = pd.read_csv(INPUT_PATH)
    df["Repository"] = df["html_url"].apply(lambda url: '/'.join(url.strip('/').split('/')[-2:])
                                            if isinstance(url, str) and '/' in url else "")
    df["nbr_of_yml"] = 0
    df["yml_detected"] = "none"
    df["CI_Service"] = "none"
    df["Valid_Repo_Step6"] = "none"
    for service in ci_services:
        if service not in df.columns:
            df[service] = 0

# === Filter Repos to Process ===
repos_to_review = df[df["Valid_Repo_Step5"] == "yes"].copy()
repos_to_review = repos_to_review[repos_to_review["Valid_Repo_Step6"] == "none"]
print(f"🔍 Reviewing {len(repos_to_review)} repos...")

# === Process Each Repo ===
for count, (idx, row) in enumerate(repos_to_review.iterrows(), start=1):
    repo = row["Repository"]
    print(f"🔎 [{count}/{len(repos_to_review)}] Checking repo: {repo}")

    if not repo or len(repo.split('/')) != 2:
        print(f"⚠️ Skipping malformed URL at index {idx}")
        continue

    try:
        # Step 1: Get default branch name
        repo_api = f"https://api.github.com/repos/{repo}"
        repo_response = requests.get(repo_api, headers=get_auth_header())
        if repo_response.status_code != 200:
            print(f"⚠️ Skipping {repo} (repo API failed {repo_response.status_code})")
            df.at[idx, "Valid_Repo_Step4"] = "no"
            continue
        default_branch = repo_response.json().get("default_branch", "master")

        # Step 2: Get SHA of latest commit on default branch
        branch_url = f"https://api.github.com/repos/{repo}/branches/{default_branch}"
        branch_response = requests.get(branch_url, headers=get_auth_header())
        if branch_response.status_code != 200:
            print(f"⚠️ Skipping {repo} (branch API failed {branch_response.status_code})")
            df.at[idx, "Valid_Repo_Step4"] = "no"
            continue
        commit_sha = branch_response.json()["commit"]["sha"]

        # Step 3: Get full tree recursively from commit SHA
        tree_url = f"https://api.github.com/repos/{repo}/git/trees/{commit_sha}?recursive=1"
        tree_response = requests.get(tree_url, headers=get_auth_header())
        if tree_response.status_code != 200:
            print(f"⚠️ Skipping {repo} (tree API failed {tree_response.status_code})")
            df.at[idx, "Valid_Repo_Step4"] = "no"
            continue

        files = tree_response.json().get("tree", []) 
        yml_count = 0
        matched_services = set()

        for file in files:
            path = file.get("path", "")
            for pattern, service in compiled_patterns:
                if pattern.search(path):
                    yml_count += 1
                    df.at[idx, service] = df.at[idx, service] + 1 if service in df.columns else 1
                    matched_services.add(service)

        if yml_count > 0:
            df.at[idx, "yml_detected"] = "yes"
            df.at[idx, "Valid_Repo_Step6"] = "yes"
            df.at[idx, "nbr_of_yml"] = yml_count
            df.at[idx, "CI_Service"] = ', '.join(sorted(matched_services))
        else:
            df.at[idx, "yml_detected"] = "no"
            df.at[idx, "Valid_Repo_Step6"] = "no"

    except Exception as e:
        print(f"❌ Error processing {repo}: {e}")
        df.at[idx, "Valid_Repo_Step6"] = "no"

    if count % SAVE_INTERVAL == 0:
        df.to_csv(OUTPUT_PATH, index=False)
        print(f"💾 Progress saved at repo #{count} (index {idx})")

    sleep(SLEEP_TIME)

# === Final Save ===
df.to_csv(OUTPUT_PATH, index=False)
print("✅ Done! All results saved to:", OUTPUT_PATH)


C:\Users\gilla\AppData\Local\Temp\ipykernel_17604\684173301.py:54: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


🔍 Reviewing 13688 repos...
🔎 [1/13688] Checking repo: wuan/bo-android
🔎 [2/13688] Checking repo: MikeOrtiz/TouchImageView
🔎 [3/13688] Checking repo: andstatus/andstatus
🔎 [4/13688] Checking repo: ubergeek42/weechat-android
🔎 [5/13688] Checking repo: persian-calendar/persian-calendar
🔎 [6/13688] Checking repo: yuriykulikov/AlarmClock
🔎 [7/13688] Checking repo: shlusiak/Freebloks-Android
🔎 [8/13688] Checking repo: traccar/traccar-client-android
🔎 [9/13688] Checking repo: DSteve595/Put.io
🔎 [10/13688] Checking repo: kevinhinterlong/archwiki-viewer
🔎 [11/13688] Checking repo: openvehicles/Open-Vehicle-Android
🔎 [12/13688] Checking repo: pilot51/voicenotify
🔎 [13/13688] Checking repo: marcoRS/nested-fragments
🔎 [14/13688] Checking repo: shadowsocks/shadowsocks-android
🔎 [15/13688] Checking repo: ankidroid/Anki-Android
🔎 [16/13688] Checking repo: anthonycr/Lightning-Browser
🔎 [17/13688] Checking repo: ligi/PassAndroid
🔎 [18/13688] Checking repo: max-kammerer/orion-viewer
🔎 [19/13688] Checkin